# Optimizing Agentic AI Collaboration v2.2.6 — Qwen3-1.7B Single-Agent Baseline
## MATH-500 only

**Model:** `Qwen/Qwen3-1.7B` with 4-bit NF4 quantization via bitsandbytes — the same load-time quantization workflow as the other SLM families in this project.
**Hardware target:** Nvidia T4 GPU (Kaggle / Colab free tier).

## Project aims

 1. a LangChain-based multi-agent Society-of-Minds prototype supporting ≥3 interaction protocols,
 2. benchmark gains (≥5% on GSM8K, MMLU and MATH-500 vs a single-agent SLM baseline), and
 3. a cost/benefit profile using latency + total token consumption, all tracked in W&B.

## What this notebook is

This notebook establishes the **single-agent baseline** for **MATH-500** using **Qwen3-1.7B**, the smallest SLM in the study. It is the reference point against which the multi-agent system (MAS) — built from instances of the *same* model — will later be compared. MATH-500 is the standard 500-problem competition-mathematics subset (`HuggingFaceH4/MATH-500`); this notebook draws a fixed 100-problem subsample — smaller than the GSM8K/MMLU baselines' (`v2.2.2`) 128 to offset the raised token budget and keep runtime bounded.

This notebook runs **a single benchmark per Kaggle session** by design, matching the split used for the other MATH-500 baselines: evaluating 100 problems under 3 generation seeds at up to 1500 tokens each fits comfortably within Kaggle's 12-hour session limit on its own, and keeping benchmarks in separate notebooks avoids any risk of exceeding it.

Unlike the v1.2.x templates, **there is no multi-agent system here**: a single direct inference call answers each problem. Consequently **no overhead ratios / deltas are computed** — those are MAS-vs-baseline comparisons and belong to the MAS notebooks.

Each of the **100 MATH-500** problems is evaluated under **3 generation seeds** to reduce estimation variability and enable cluster-robust confidence intervals.

## Relation with the project WPs

**WP2 - T3: Single-agent baseline establishment (CoT).** This notebook establishes a zero-shot, CoT, single-agent baseline for one of the discuted state-of-the-art SLMs (Qwen3-1.7B) over MATH-500. This baseline is the reference the L1/L2 multi-agent protocols (WP2 - T4/T6) must overcome, and its latency/token/VRAM measurements feed the WP3 - T1 compute-accuracy trade-off analysis.

---
### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel (or open it directly if already there).
1. In **Settings**, enable **Accelerator: GPU T4 x2** (the notebook pins itself to one GPU).
2. Add the following secrets via **Add-ons → Secrets**, enabling each for the notebook:
   - `GITHUB_TOKEN` — GitHub PAT with repo read scope
   - `HF_TOKEN` — Hugging Face read-access token
   - `WANDB_API_KEY` — Weights & Biases API key
3. Click **Run All**. The first cell auto-detects Kaggle, clones the repo, and loads secrets.

### Before running (Colab)

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets** (🔑 icon in the left sidebar):
   - `HF_TOKEN` — your Hugging Face read-access token ([create one here](https://huggingface.co/settings/tokens))
   - `WANDB_API_KEY` — your Weights & Biases API key ([find it here](https://wandb.ai/authorize))
   - `GITHUB_TOKEN` — a GitHub Personal Access Token with `repo` read scope
   - **Enable "Notebook access"** for all secrets

## What this notebook implements

A **single-agent baseline** over **MATH-500** with W&B tracking:

* A single direct-inference call per problem (no Proposer/Critic/Refiner pipeline).
* **W&B logging** of per-call latency, output/total token counts, GPU peak VRAM, and generation throughput (control metric); plus a unified W&B `Table` of every sample.
* Each problem is run under **3 generation seeds**; correctness, latency, token and peak-VRAM matrices (N×m) are persisted to a W&B artifact for downstream analysis.
* **Accuracy, latency, output/total tokens and peak VRAM** are summarized with a point estimate **and a cluster-robust 95% confidence interval**.
* Correctness is scored with `math_match` (`math-verify`, sympy-based LaTeX/symbolic equivalence) rather than the numeric matcher used for GSM8K, since MATH-500 answers are LaTeX expressions (fractions, roots, intervals, ...).

This produces a robust single-agent baseline accuracy and computational cost (OBJ-2,3) with a SOTA SLM, on which the MAS comparison will build. Overhead ratios/deltas are **not** computed here.

## 1. Setup and Installation

We install the libraries required by the repo (Hugging Face stack, `langchain*`, `datasets`, `wandb`, `math-verify`), all pinned in `requirements.txt`. If running on Colab/Kaggle, the repo (with the `utils` helper modules) is cloned first.

In [1]:
# ── Environment detection: clone private repo ────────────────
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

Cloning into '/kaggle/working/tfg-agentic-ai'...


✅ Repo cloned to /kaggle/working/tfg-agentic-ai
repo_root = /kaggle/working/tfg-agentic-ai


In [2]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%cd {repo_root}
%pip install -r requirements.txt

/kaggle/working/tfg-agentic-ai
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.9/211.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.

In [3]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

Runtime environment: kaggle
✅ Secrets loaded from Kaggle Secrets.


## 3. Define Runs Metadata

We define the W&B project and configuration metadata, in order to later create a W&B run that will store all the baseline calls, metrics and statistics of the model's execution.

In [4]:
RUN_NAME = "Baseline-Qwen3-1.7B"

PROJECT = "baselines-results" if (ENVIRONMENT == "kaggle") else "dev-agentic-baselines"
print(f"W&B Project: {PROJECT}")

GEN = {
    "max_new_tokens": 1500,
    "temperature":    0.7,
    "top_p":          0.95,
    "top_k":          20,
}

CONFIG = {
        "model_id":         "Qwen/Qwen3-1.7B",
        "quantization":     "nf4-4bit",
        "framework":        "langchain",
        "protocol":         "single_agent_baseline",
        "eval_size":        100,
        "data_seed":        789,
        "generation_seeds": [42, 123, 456],
    }

W&B Project: baselines-results


## 4. Data Loading
We load the MATH-500 (`HuggingFaceH4/MATH-500`) dataset used by the evaluation section below.

In [5]:
from utils.data import load_math500

# ── MATH-500 DATASET ──────────────────────────────────────────────────
math500_dataset = load_math500(split="test")

print(f"✅ MATH-500 dataset loaded with {len(math500_dataset)} examples.")
print(f"Example problem: {math500_dataset[0]['problem']}")
print(f"Example answer:  {math500_dataset[0]['answer']}")
print(f"Example level:   {math500_dataset[0]['level']}  (subject: {math500_dataset[0]['subject']})")

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ MATH-500 dataset loaded with 366 examples.
Example problem: Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$
Example answer:  \left( 3, \frac{\pi}{2} \right)
Example level:   2  (subject: Precalculus)


## 5. Model Loading (4-bit Quantized)

Qwen3-1.7B is a standard causal LM (`Qwen3ForCausalLM`) shipping in **BF16**. `load_model_and_tokenizer` quantizes it to **NF4 4-bit** on the fly via bitsandbytes (~1.5 GB on a T4) using the standard `AutoModelForCausalLM` path — no special dispatch, no `trust_remote_code`, and no `mistral-common` dependency. The tokenizer is a standard HuggingFace fast tokenizer and decodes correctly without any extra configuration.

In [6]:
import wandb
from utils.models import load_model_and_tokenizer

model_id = CONFIG["model_id"]
model, tokenizer = load_model_and_tokenizer(model_id, quantize_4bit=True)

print(f"✅ Model loaded on {model.device} — dtype {model.dtype}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ Model loaded on cuda:0 — dtype torch.float16


## 6. Defining the Single-Agent Baseline

The baseline is a single LangChain chain: a `PromptTemplate` piped to the LLM (execution engine). No Proposer/Critic/Refiner — just one direct call per problem. We define the MATH-500 template (EvalScope's competition-math wording, box-notation final answer), used by `run_single_agent_baseline`. Qwen3 requires the `/no_think` directive to disable its thinking mode for this turn, matching the GSM8K/MMLU baselines.

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Parser
parser = StrOutputParser()

# ── MATH-500 Template ────────────────────────────────────
# /no_think disables Qwen3's thinking mode (direct answer + visible CoT in response text).
math_template = PromptTemplate.from_template(
        "/no_think\n\n"
        "Problem:\n{task}\n\n"
        "Please reason step by step, and put your final answer within \\boxed{{}}."
)

print("✅ MATH-500 Single-Agent chain template defined.")

✅ MATH-500 Single-Agent chain template defined.


## 7. Single-Agent Runner with W&B Logging

`run_agent_step` times a single `chain.invoke`, counts output/total tokens with the model's own tokenizer (outside the timed region so latency stays clean), and measures per-call peak VRAM. `run_single_agent_baseline` wraps one baseline call against the MATH-500 template, logging latency, tokens, peak VRAM and throughput per sample. **No overhead ratios are computed** — this is a baseline, not a MAS-vs-baseline comparison.

In [8]:
import time
from utils.logging_utils import log_metrics
from utils.instrumentation import (
    count_tokens, render_prompt,
    reset_gpu_peak, gpu_peak_mem_mb,
    tokens_per_second,
)

def run_agent_step(chain, inputs, step_name, prompt_template):
    """Run a single LangChain chain, time it, count tokens, measure per-step peak VRAM and throughput (tokens/s).

    Token counting and VRAM measurement are performed outside the timed region
    so the latency measurement (and the throughput one) wraps only ``chain.invoke``.

    Returns
    -------
    tuple[str, float, int, int, float]
        (output_text, elapsed_s, input_tokens, output_tokens, peak_vram_mb)
    """
    # Input tokens: counted BEFORE the timed region
    prompt_str   = render_prompt(prompt_template, inputs)
    input_tokens = count_tokens(tokenizer, prompt_str)

    # Reset peak VRAM counter for this step's independent measurement
    reset_gpu_peak()

    # Timed region: pure chain.invoke only
    t0          = time.perf_counter()
    output_text = chain.invoke(inputs)
    elapsed     = time.perf_counter() - t0

    # Post-invoke measurements: outside the timed region
    output_tokens = count_tokens(tokenizer, output_text)
    total_tokens  = input_tokens + output_tokens
    peak_vram_mb  = gpu_peak_mem_mb()
    tps           = tokens_per_second(output_tokens, elapsed)

    print(
        f"  ⏱  {step_name}: {elapsed:.1f}s  |  "
        f"{len(output_text)} chars  |  "
        f"in={input_tokens} out={output_tokens} tok  |  "
        f"peak={peak_vram_mb:.0f} MB  |  {tps:.1f} tok/s"
    )
    return output_text, elapsed, input_tokens, output_tokens, peak_vram_mb


def run_single_agent_baseline(chain, task, task_label="math_sample"):
    """Execute a single-agent baseline call against the MATH-500 template, with W&B
    logging of latency, tokens, GPU peak VRAM and throughput. No multi-agent
    pipeline and no overhead metrics.

    Parameters
    ----------
    chain     : LangChain chain built for the current generation seed.
    task      : Formatted MATH-500 problem string.
    task_label: W&B metric prefix (e.g. ``"math_sample"``).

    Returns
    -------
    dict with keys:
        baseline                                   (text output)
        t_base                                     (latency, s)
        base_in_tok, base_out_tok, base_total_tok  (baseline tokens)
        base_peak_mb                               (peak VRAM, MB)
        base_throughput                            (tok/s)
    """
    print(f"\n{'='*60}")
    print(f"📊 SINGLE-AGENT BASELINE  —  {task_label}")
    print(f"{'='*60}")

    baseline, t_base, in_base, out_base, peak_base = run_agent_step(
        chain, {"task": task},
        f"{task_label}/baseline", math_template
    )

    base_total_tok  = in_base + out_base
    base_throughput = tokens_per_second(out_base, t_base)

    # ── Log per-sample baseline summary to W&B (no overhead metrics) ──
    log_metrics({
        f"{task_label}/latency_s":      round(t_base, 2),
        f"{task_label}/output_tokens":  out_base,
        f"{task_label}/total_tokens":   base_total_tok,
        f"{task_label}/gpu_peak_mb":    round(peak_base, 2),
        f"{task_label}/throughput_tps": round(base_throughput, 2),
    })

    return {
        "baseline":        baseline,
        "t_base":          t_base,
        "base_in_tok":     in_base,
        "base_out_tok":    out_base,
        "base_total_tok":  base_total_tok,
        "base_peak_mb":    peak_base,
        "base_throughput": base_throughput,
    }

print("✅ Single-agent runner ready.")

✅ Single-agent runner ready.


## 8. Model Warmup
Before timing, we "warm up" the model with 2 dummy executions so the one-time CUDA / bitsandbytes kernel compilation and cuBLAS autotuning happen here — not inside the first timed sample.

In [9]:
# ── Warmup ────────────────────────────────────────────────────────────
# Run a couple of throwaway generations through the same pipeline the
# evaluation uses, so one-time CUDA / bitsandbytes kernel compilation and
# cuBLAS autotuning happen here — not inside the first timed sample.
from utils.warmup import warmup_model

warmup_model(
    model, tokenizer,
    max_new_tokens=256, n_iterations=2,
    temperature=GEN["temperature"],
    top_p=GEN["top_p"], top_k=GEN["top_k"],
)

[transformers] Passing `generation_config` together with generation-related arguments=({'top_k', 'do_sample', 'top_p', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🔥 Warming up model (2 iterations, max_new_tokens=256)…


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Warmup complete in 39.9s — first evaluated sample will no longer pay cold-start cost.


39.93098908300004

## 9. MATH-500 — Initialize Weights & Biases

We open a dedicated W&B run for the MATH-500 evaluation. All MATH-500 baseline calls, latencies, and outputs will be logged here. The run is closed at the end of section 11.

In [10]:
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics

run = init_wandb(
    run_name=f"MATH500-{RUN_NAME}",
    project=PROJECT,
    config={**CONFIG, **GEN, "eval_dataset": "math500"}
)
print(f"✅ W&B run: {run.name}  —  {run.url}")

define_wandb_chart_only_metrics()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: andreu-fernandez-cubi (tfg-optimizing-agentic-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 3ocw61ao
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/tfg-agentic-ai/wandb/run-20260721_222217-3ocw61ao
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run MATH500-Baseline-Qwen3-1.7B
wandb: ⭐️ View project at https://wandb.ai/tfg-optimizing-agentic-ai/baselines-results
wandb: 🚀 View run at https://wandb.ai/tfg-optimizing-agentic-ai/baselines-results/runs/3ocw61ao


✅ W&B run: MATH500-Baseline-Qwen3-1.7B  —  https://wandb.ai/tfg-optimizing-agentic-ai/baselines-results/runs/3ocw61ao


## 10. MATH-500 Evaluation

We evaluate the single-agent baseline over the fixed 100-problem subset, repeating each problem under 3 generation seeds. For each `(problem, seed)` we record correctness, latency, token counts and peak VRAM. Correctness is scored by `math_match` on the **full** model output — `math-verify`'s own boxed/LaTeX extraction is more robust than re-cutting the string first; `extract_math_prediction` only fills the `final_output` display column.

In [11]:
import warnings
warnings.filterwarnings("ignore") # to ignore transformer warnings that can clutter the output during evaluation

import random
import numpy as np
from tqdm.notebook import tqdm
from utils.logging_utils import log_metrics
from utils.agents import build_hf_langchain_llm
from utils.evaluation import extract_math_prediction, math_match
from langchain_core.output_parsers import StrOutputParser

generation_seeds = wandb.config.generation_seeds
sample_size      = wandb.config.eval_size

# Fix the evaluation subset once — all seed runs evaluate the same problems
random.seed(wandb.config.data_seed)
test_sample = random.sample(list(math500_dataset), sample_size)

# ── Baseline matrices: rows = problems, cols = seeds ───────────────────
baseline_correctness_matrix = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_time_matrix        = np.zeros((sample_size, len(generation_seeds)), dtype=float)
baseline_input_tok_matrix   = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_output_tok_matrix  = np.zeros((sample_size, len(generation_seeds)), dtype=int)
baseline_gpu_peak_matrix    = np.zeros((sample_size, len(generation_seeds)), dtype=float)

# ── Single unified results table accumulated across ALL seeds ─────────────
results_columns = [
    "model_id", "dataset", "protocol",
    "problem_id", "seed",
    "task",
    "output",
    "ground_truth", "final_output",
    "latency_s",
    "output_tokens", "total_tokens",
    "gpu_peak_mb",
    "throughput_tok_s",
    "subject", "level",
]
results_rows = []

for run_idx, gen_seed in enumerate(generation_seeds):
    print(f"\n{'='*60}")
    print(f"  Seed run {run_idx + 1}/{len(generation_seeds)}  —  generation_seed={gen_seed}")
    print(f"{'='*60}")

    # Rebuild LLM and the baseline chain with the current generation seed
    llm = build_hf_langchain_llm(
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=wandb.config.max_new_tokens,
        temperature=wandb.config.temperature,
        top_k=wandb.config.top_k,
        top_p=wandb.config.top_p,
        return_full_text=False,
        seed=gen_seed,
    )
    parser = StrOutputParser()
    baseline_chain = math_template | llm | parser

    for item_idx, item in enumerate(tqdm(test_sample, desc=f"MATH500 seed={gen_seed} ({sample_size} samples)")):
        task        = item['problem']
        true_answer = item['answer']

        output = None
        try:
            output            = run_single_agent_baseline(baseline_chain, task, task_label='math_sample')
            baseline_response = output['baseline']
        except Exception as e:
            print(f"Baseline failed: {e}")
            baseline_response = str(e)
            if output is None:
                output = {
                    'baseline': baseline_response,
                    't_base': 0.0,
                    'base_in_tok': 0, 'base_out_tok': 0, 'base_total_tok': 0,
                    'base_peak_mb': 0.0, 'base_throughput': 0.0,
                }

        # Correctness is scored on the FULL model output — math_verify performs
        # its own boxed/LaTeX extraction, which is more robust than re-cutting
        # the string first. extract_math_prediction is display-only (below).
        baseline_answer  = extract_math_prediction(baseline_response)
        baseline_correct = math_match(baseline_response, true_answer)

        # ── Fill baseline matrices ────────────────────────────────────
        baseline_correctness_matrix[item_idx, run_idx] = int(baseline_correct)
        baseline_time_matrix[item_idx, run_idx]        = output['t_base']
        baseline_input_tok_matrix[item_idx, run_idx]   = output['base_in_tok']
        baseline_output_tok_matrix[item_idx, run_idx]  = output['base_out_tok']
        baseline_gpu_peak_matrix[item_idx, run_idx]    = output['base_peak_mb']

        log_metrics({
            "math_sample/accuracy": int(baseline_correct),
        })

        # Accumulate one row for the unified results table (problem_id, seed first)
        results_rows.append([
            CONFIG['model_id'], "math500", CONFIG['protocol'],
            item_idx, gen_seed,
            task,
            output['baseline'][:2000],
            true_answer, baseline_answer,
            round(output['t_base'], 2),
            output['base_out_tok'], output['base_total_tok'],
            round(output['base_peak_mb'], 2),
            round(output['base_throughput'], 4),
            item['subject'], item['level'],
        ])

# ── Log the unified results table ONCE, sorted by problem_id then seed ─────
results_rows.sort(key=lambda r: (r[3], r[4]))
results_table = wandb.Table(columns=results_columns, data=results_rows)
wandb.log({f"results_comparison_MATH500": results_table})


  Seed run 1/3  —  generation_seed=42


MATH500 seed=42 (100 samples):   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📊 SINGLE-AGENT BASELINE  —  math_sample
  ⏱  math_sample/baseline: 112.7s  |  3202 chars  |  in=67 out=1433 tok  |  peak=1584 MB  |  12.7 tok/s


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  ⏱  math_sample/baseline: 51.3s  |  1451 chars  |  in=101 out=632 tok  |  peak=1480 MB  |  12.3 tok/s

📊 SINGLE-AGENT BASELINE  —  math_sample


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ⏱  math_sample/baseline: 26.2s  |  778 chars  |  in=45 out=326 tok  |  peak=1417 MB  |  12.4 tok/s

📊 SINGLE-AGENT BASELINE  —  math_sample


[transformers] Both `max_new_tokens` (=1500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ⏱  math_sample/baseline: 29.4s  |  1081 chars  |  in=64 out=369 tok  |  peak=1427 MB  |  12.5 tok/s

📊 SINGLE-AGENT BASELINE  —  math_sample
  ⏱  math_sample/baseline: 6.4s  |  170 chars  |  in=42 out=79 tok  |  peak=1393 MB  |  12.4 tok/s


## 11. MATH-500 — Results, Statistics & Finish W&B Run

We compute cluster-robust statistics for the baseline (accuracy, latency, output/total tokens, peak VRAM), push them and the results table to W&B, persist the matrices as a W&B artifact, and close the MATH-500 run.

In [12]:
import tempfile
import wandb
import numpy as np
from utils.logging_utils import finish_wandb, log_stat_summary
from utils.statistics import (
    accuracy_statistics,
    time_statistics,
    token_statistics,
    vram_statistics,
    print_accuracy_stats_report,
    print_time_report,
    print_token_report,
    print_vram_report,
)

DATASET = "MATH500"

# ── Derived matrix ───────────────────────────────────────────────────────
baseline_total_tok_matrix = baseline_input_tok_matrix + baseline_output_tok_matrix

# ── Accuracy ─────────────────────────────────────────────────────────────
baseline_acc_stats = accuracy_statistics(baseline_correctness_matrix)
print_accuracy_stats_report(f"Baseline Accuracy ({DATASET})", baseline_acc_stats)

# ── Latency ──────────────────────────────────────────────────────────────
baseline_time_stats = time_statistics(baseline_time_matrix)
print_time_report(f"Baseline Latency ({DATASET})", baseline_time_stats)

# ── Output token counts ───────────────────────────────────────────────────
baseline_out_tok_stats = token_statistics(baseline_output_tok_matrix, max_tokens=wandb.config.max_new_tokens)
print_token_report(f"Baseline Output Tokens ({DATASET})", baseline_out_tok_stats)

# ── Total token counts (input + output) ─────────────────────────────────
baseline_total_tok_stats = token_statistics(baseline_total_tok_matrix)
print_token_report(f"Baseline Total Tokens ({DATASET})", baseline_total_tok_stats)

# ── Peak VRAM ─────────────────────────────────────────────────────────────
baseline_vram_stats = vram_statistics(baseline_gpu_peak_matrix)
print_vram_report(f"Baseline Peak VRAM ({DATASET})", baseline_vram_stats)

# ── W&B summary ──────────────────────────────────────────────────────────
wandb.summary["n_seeds"]           = len(generation_seeds)
wandb.summary["n_problems"]        = sample_size
wandb.summary["total_experiments"] = sample_size * len(generation_seeds)

# Baseline: accuracy statistics
wandb.summary["accuracy"] = round(baseline_acc_stats["p_hat"], 6)
log_stat_summary("accuracy_stats", baseline_acc_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "wilson_ci_lower", "wilson_ci_upper", "wilson_n_eff", "wilson_icc"])

# Baseline: latency statistics
wandb.summary["latency"] = round(baseline_time_stats["mean"], 2)
log_stat_summary("latency_stats", baseline_time_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "flat_cv"])

# Baseline: output token statistics
wandb.summary["output_tokens"] = round(baseline_out_tok_stats["mean"], 1)
log_stat_summary("output_tokens_stats", baseline_out_tok_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index", "frac_at_ci_lower", "frac_at_ci_upper"])

# Baseline: total token statistics
wandb.summary["total_tokens"] = round(baseline_total_tok_stats["mean"], 1)
log_stat_summary("total_tokens_stats", baseline_total_tok_stats,
    keys=["ci_lower", "ci_upper", "between_problem_std", "dispersion_index"])

# Baseline: peak VRAM statistics
wandb.summary["mean_vram_peak"] = round(baseline_vram_stats["mean_peak"], 0)
wandb.summary["max_vram_peak"]  = round(baseline_vram_stats["peak_max"], 0)
log_stat_summary("vram_peak_stats", baseline_vram_stats,
    keys=["peak_ci_lower", "peak_ci_upper", "between_problem_peak_std"])


# ── Upload all matrices as W&B artifact ──────────────────────────────────
artifact = wandb.Artifact(
    f"evaluation-matrices-Qwen-3-1.7B-Baseline-{DATASET}",
    type="evaluation-data",
    description=(
        f"Single-agent baseline evaluation matrices ({len(generation_seeds)} x {sample_size}). {DATASET} evaluation with {CONFIG['model_id']}."
        "Includes: correctness, execution times, input/output/total token counts, "
        "GPU peak VRAM (MB)."
    ),
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(
        f.name,
        baseline_correctness=baseline_correctness_matrix,
        baseline_times=baseline_time_matrix,
        baseline_input_tokens=baseline_input_tok_matrix,
        baseline_output_tokens=baseline_output_tok_matrix,
        baseline_total_tokens=baseline_total_tok_matrix,
        baseline_gpu_peak=baseline_gpu_peak_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)
print(f"✅ All evaluation matrices uploaded as W&B artifact.")

finish_wandb()
print("W&B run finished. View your dashboard at the URL above.")


  Baseline Accuracy (MATH500)
  Accuracy p̂                  71.33%
  95% Cluster-t CI         [63.69, 78.97]%
  Between-problem std        38.4958%
  ── Supplementary: Wilson ──
  95% Wilson CI              [63.29, 78.22]%
  DEFF                       137.988
  ICC                          0.587
  ── Sample ──
  Problems x Seeds           100 x 3  (n=300)


  Baseline Latency (MATH500)
  Mean (s)                     51.2732
  Std (s)                      32.2202
  CV                            0.6284
  95% BCa CI (s)             [45.6285, 57.5599]  [cluster_bca]
  Between-problem std (s)      30.5374
  ── Percentiles (descriptive) ──
  Median / P50 (s)             41.9343
  P25 (s)                      24.7650
  P75 (s)                      68.2195
  IQR (s)                      43.4545
  P95 (s)                     117.2360
  ── Sample ──
  Problems x Seeds           100 x 3  (n=300)


  Baseline Output Tokens (MATH500)
  Mean tok                        651.0800
  Std tok           

wandb: uploading artifact run-3ocw61ao-results_comparison_MATH500; updating run metadata; uploading artifact evaluation-matrices-Qwen-3-1.7B-Baseline-MATH500
wandb: uploading artifact run-3ocw61ao-results_comparison_MATH500; uploading artifact evaluation-matrices-Qwen-3-1.7B-Baseline-MATH500
wandb: uploading media/table/results_comparison_MATH500_600_5892c2d3bf90589a70f1.table.json; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:       math_sample/accuracy ██▁██▁█▁▁████▁▁▁█▁█████▁█▁█▁████████████
wandb:    math_sample/gpu_peak_mb █▂▂▄▂█▂▅▂▆▄▆▄▄▂▂▄▅▄▂▆█▄▄▅▁▂▄▁▂▇▂▇▅▂▂▇▅▄▄
wandb:      math_sample/latency_s █▃▃▄▇▂▄▂▂▂▂▅▃▂█▄▂▅▂█▂█▅▅▂▆▃▂▂▂▄▂█▆▂▂▃█▃▁
wandb:  math_sample/output_tokens ▂▂█▁▁▁▃▃▄█▁▂▃█▂▂▂▆▃▁▂▁█▃▄▁▁▆▄█▁▂██▁▂▅▄▄▂
wandb: math_sample/throughput_tps █▆▆▇▅▇█▇▇▇▇▇▇▆▇▇█▆▇▅▇▆▇████▇▁▇█▇▅▆▇▇▆▇▇▅
wandb:   math_sample/total_tokens ▄▅▁▃▂▄█▂▅▅▄█▃▃▅▆▂▂▃▅▅▂▆▃▃▅▄▅▅▃█▂▁▃▃▂▃▃▃▃
wandb: 
wandb: Run summary:
wandb:                           a

W&B run finished. View your dashboard at the URL above.


## 12. Analysis & Next Steps

### What W&B gives you (this run)
- **Correctness / accuracy** — per-sample and aggregate baseline accuracy with a 95% CI, scored via `math-verify` LaTeX/symbolic equivalence rather than a numeric or string match.
- **Latency, token and VRAM panels** — per-sample `math_sample/baseline_*` metrics and their cluster-robust summaries.
- **Results table** — every sample's prompt, output, predicted vs. ground-truth expression, latency, tokens, peak VRAM, throughput, and the problem's `subject`/`level` (1–4) for later per-level breakdowns.
- **Artifact** — `evaluation_matrices.npz` with the baseline correctness / latency / token / VRAM matrices (N×m), for downstream statistical analysis.

### Expected results
MATH-500 is harder than GSM8K, especially at level 4; accuracy for a 1.7B model is expected to be markedly lower here than on the GSM8K/MMLU baselines (`v2.2.2`). `max_new_tokens=1500` (raised above the GSM8K/MMLU baselines' 1024 to better accommodate long competition-math derivations) can still truncate the rare long-tail solution before it reaches `\boxed{}`, which scores as incorrect — the truncation rate is visible via `output_tokens_stats/frac_at_ci_upper` in this run's summary. Level-5 problems are excluded from the sampled pool (`utils/data.py::load_math500`) since their reasoning chains are especially prone to this truncation.

### Role of this notebook
This is the **single-agent baseline** for MATH-500 with Qwen3-1.7B, the smallest SLM in the study — it anchors the small end of the accuracy-vs-size sweep alongside the Qwen3-4B and Qwen3-8B MATH-500 baselines. **Overhead ratios/deltas are intentionally not computed here** — the MAS notebooks will load these matrices and compute MAS-vs-baseline overhead against them.

### Next steps
1. Build the multi-agent system from instances of this same model and compare accuracy and cost (latency + total tokens + VRAM) against these baselines (OBJ-1,2,3), including a per-level (1–4) accuracy breakdown using the `level` column in the results table.